# LeetCode 1628 — Design an Expression Tree With Evaluate Function bằng Python

Link đề: https://leetcode.com/problems/design-an-expression-tree-with-evaluate-function/ · Độ khó: **Medium** · Chủ đề: Array, Math, Stack, Tree, Design, Binary Tree

> **Bài Premium.** Đề trong notebook này được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại câu chữ, ví dụ và đặc biệt **quy ước phép chia** với đề gốc.

Skeleton LeetCode (Python3):

```python
import abc
from abc import ABC, abstractmethod
"""
This is the interface for the expression tree Node.
You should not remove it, and you can define some classes to implement it.
"""

class Node(ABC):
    @abstractmethod
    # define your fields here
    def evaluate(self) -> int:
        pass

"""
This is the TreeBuilder class.
You can treat it as the driver code that takes the postinfix input
and returns the expression tree represnting it as a Node.
"""

class TreeBuilder(object):
    def buildTree(self, postfix: List[str]) -> 'Node':
        ...

"""
Your TreeBuilder object will be instantiated and called as such:
obj = TreeBuilder();
expTree = obj.buildTree(postfix);
ans = expTree.evaluate();
"""
```

## 1. Mục tiêu học

1. Hiểu **ký pháp hậu tố** (postfix / Reverse Polish Notation) và vì sao nó không cần dấu ngoặc.
2. Dựng **cây biểu thức nhị phân** từ postfix bằng **stack** (và bằng đệ quy đọc từ phải sang trái).
3. Thiết kế lớp theo hướng đối tượng: **lớp trừu tượng** `Node` (module `abc`), lớp con cho số và cho phép toán, **đa hình** cho `evaluate()`.
4. Trả lời câu follow-up: thiết kế **mở rộng được** — thêm phép toán mới (ví dụ `%`, `^`) mà **không sửa** `evaluate` cũ (nguyên lý Open/Closed).
5. Viết bộ chạy giả lập cách LeetCode gọi `TreeBuilder().buildTree(postfix).evaluate()`, cùng test ngẫu nhiên đối chiếu với bộ tính postfix trực tiếp.

## 2. Đề bài & ví dụ

Cho mảng token **hậu tố** `postfix` của một biểu thức số học, hãy **dựng và trả về cây biểu thức nhị phân** biểu diễn biểu thức đó.

- Ký pháp hậu tố: toán hạng đứng **trước** toán tử. Ví dụ `4*(5-(7+2))` có postfix `["4","5","7","2","+","-","*"]`.
- Phải dùng **interface `Node`** (không được xóa nó, nhưng được sửa / định nghĩa thêm lớp). Cây trả về được chấm bằng cách gọi `evaluate()` trên gốc.
- Cây biểu thức nhị phân: mỗi nút có **0 hoặc 2 con**. **Lá** là toán hạng (số), **nút trong** là toán tử `+`, `-`, `*`, `/`.
- Đảm bảo: không cây con nào có giá trị tuyệt đối vượt `10⁹`, không chia cho 0.

**Follow up:** thiết kế sao cho **mô-đun hơn** — có thể hỗ trợ thêm toán tử mà **không phải sửa** cài đặt `evaluate` hiện có?

**Ví dụ 1:** `s = ["3","4","+","2","*","7","/"]` → `2` vì `((3+4)*2)/7 = 14/7 = 2`.

**Ví dụ 2:** `s = ["4","5","2","7","+","-","*"]` → `-16` vì `4*(5-(2+7)) = 4*(-4) = -16`.

**Ràng buộc:**
- `1 ≤ s.length < 100`, `s.length` lẻ
- `s` gồm các số và các ký tự `+`, `-`, `*`, `/`
- Mỗi số không vượt `10⁵`
- `s` là biểu thức hợp lệ; kết quả và mọi giá trị trung gian có trị tuyệt đối `≤ 10⁹`; không chia cho 0.

**Về phép chia:** đề gốc (thiết kế cho mọi ngôn ngữ) dùng **chia nguyên**. Trong C++/Java chia nguyên **làm tròn về 0** (`-7/2 = -3`), còn `//` của Python **làm tròn xuống** (`-7//2 = -4`). Notebook dùng **làm tròn về 0** cho an toàn và chỉ ra chỗ khác biệt ở mục 13 — hãy đối chiếu lại đề gốc nếu có Premium.

In [1]:
import random
import time
import sys
from abc import ABC, abstractmethod
from typing import List

examples = [
    (["3", "4", "+", "2", "*", "7", "/"], 2),
    (["4", "5", "2", "7", "+", "-", "*"], -16),
]
for s, want in examples:
    print(" ".join(s), "-> kỳ vọng", want)

3 4 + 2 * 7 / -> kỳ vọng 2
4 5 2 7 + - * -> kỳ vọng -16


## 3. Kiến thức nền

### 3.1 Ký pháp hậu tố và tính bằng stack

Đọc token từ trái sang phải: gặp **số** thì đẩy vào stack; gặp **toán tử** thì lấy ra **hai** phần tử trên cùng — phần tử lấy ra **trước là toán hạng phải**, lấy ra **sau là toán hạng trái** — tính rồi đẩy kết quả lại. Cuối cùng stack còn đúng 1 phần tử. Không cần ngoặc vì thứ tự token đã mã hóa thứ tự tính.

In [2]:
def div_trunc(a, b):
    """Chia nguyên làm tròn về 0 (như C++/Java)."""
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

OPS = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": div_trunc,
}

def eval_postfix(tokens):
    st = []
    for t in tokens:
        if t in OPS:
            right = st.pop(); left = st.pop()
            st.append(OPS[t](left, right))
        else:
            st.append(int(t))
    assert len(st) == 1
    return st[0]

for s, want in examples:
    print(eval_postfix(s), "kỳ vọng", want)
print("div_trunc:", div_trunc(7, 2), div_trunc(-7, 2), div_trunc(7, -2), div_trunc(-7, -2), "| Python //:", -7 // 2)

2 kỳ vọng 2
-16 kỳ vọng -16
div_trunc: 3 -3 -3 3 | Python //: -4


### 3.2 Lớp trừu tượng với `abc`

`class Node(ABC)` + `@abstractmethod def evaluate(...)` nghĩa là: `Node` là **interface**, không tạo đối tượng trực tiếp được; mọi lớp con **phải** cài `evaluate`, nếu không cũng không tạo được.

In [3]:
class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Square(Shape):
    def __init__(self, a): self.a = a
    def area(self): return self.a * self.a

class Broken(Shape):          # quên cài area
    pass

print(Square(3).area())
for cls in (Shape, Broken):
    try:
        cls()
    except TypeError as e:
        print(f"không tạo được {cls.__name__}: {type(e).__name__}")

9
không tạo được Shape: TypeError
không tạo được Broken: TypeError


### 3.3 Đa hình (polymorphism)

Gọi `x.evaluate()` mà không cần biết `x` là lá hay nút toán tử — mỗi lớp tự biết cách tính. Đây là chìa khóa cho follow-up: thêm loại nút mới = thêm lớp mới, **không sửa code cũ**.

In [4]:
class Animal(ABC):
    @abstractmethod
    def sound(self): ...
class Dog(Animal):
    def sound(self): return "gâu"
class Cat(Animal):
    def sound(self): return "meo"
print([a.sound() for a in (Dog(), Cat(), Dog())])

['gâu', 'meo', 'gâu']


### 3.4 Duyệt cây: hậu thứ tự (postorder) và trung thứ tự (inorder)

- **Postorder** (trái → phải → gốc) của cây biểu thức chính là postfix.
- **Inorder có ngoặc** cho biểu thức dạng quen thuộc `((3+4)*2)/7`.
Hai hàm này giúp **kiểm tra** cây ta dựng: postorder phải ra đúng mảng đầu vào.

In [5]:
class T:                       # cây đơn giản để minh họa
    def __init__(self, v, l=None, r=None): self.v, self.l, self.r = v, l, r

def post(t): return [] if t is None else post(t.l) + post(t.r) + [t.v]
def infix(t): return t.v if t.l is None else f"({infix(t.l)}{t.v}{infix(t.r)})"

tree = T("/", T("*", T("+", T("3"), T("4")), T("2")), T("7"))
print(post(tree)); print(infix(tree))

['3', '4', '+', '2', '*', '7', '/']
(((3+4)*2)/7)


## 4. Ý tưởng nền tảng / trực giác

Dựng cây cho ví dụ 2 `["4","5","2","7","+","-","*"]` bằng stack — thay vì đẩy **giá trị**, ta đẩy **nút cây**:

| token | hành động | stack (đáy → đỉnh) |
|---|---|---|
| 4 | lá | `4` |
| 5 | lá | `4, 5` |
| 2 | lá | `4, 5, 2` |
| 7 | lá | `4, 5, 2, 7` |
| + | pop `7` (phải), pop `2` (trái) → `(2+7)` | `4, 5, (2+7)` |
| - | pop `(2+7)`, pop `5` → `(5-(2+7))` | `4, (5-(2+7))` |
| * | pop, pop → `(4*(5-(2+7)))` | `(4*(5-(2+7)))` |

Phần tử duy nhất còn lại là **gốc**. Tính: `2+7 = 9`, `5−9 = −4`, `4·(−4) = −16` ✔.

Nhận xét quan trọng: **token cuối của postfix luôn là gốc**; ngay trước nó là postfix của **cây con phải**, phía trước nữa là postfix của **cây con trái** ⇒ cho phép dựng đệ quy từ phải sang trái (Cách 3).

## 5. Cách 1 — Brute Force / baseline: tính trực tiếp postfix (không dựng cây)

**Ý tưởng:** chỉ cần ra **giá trị** thì stack số ở mục 3.1 là đủ — `O(n)`. Nhưng bài là bài **thiết kế**: LeetCode gọi `buildTree` rồi `evaluate()` trên đối tượng trả về. Ta có thể "lách" bằng một nút giả lưu sẵn đáp án:

In [6]:
class Node(ABC):
    @abstractmethod
    def evaluate(self) -> int:
        pass

class PrecomputedNode(Node):
    """Baseline: 'cây' chỉ có 1 nút giữ sẵn kết quả."""
    def __init__(self, value):
        self.value = value
    def evaluate(self) -> int:
        return self.value

class TreeBuilderBaseline:
    def buildTree(self, postfix: List[str]) -> Node:
        return PrecomputedNode(eval_postfix(postfix))

def run_leetcode(builder_cls, postfix):
    """Mô phỏng driver của LeetCode."""
    obj = builder_cls()
    expTree = obj.buildTree(postfix)
    return expTree.evaluate()

for s, want in examples:
    print(run_leetcode(TreeBuilderBaseline, s), "kỳ vọng", want)

2 kỳ vọng 2
-16 kỳ vọng -16


**Độ phức tạp:** Time `O(n)`, Space `O(n)` với `n = len(postfix)`.

**Nút thắt:** đây **không phải** cây biểu thức — không thể in lại biểu thức, không thể đổi giá trị một lá rồi tính lại, không mở rộng được. Nó đóng vai **oracle** (bộ tính đúng) để kiểm thử các thiết kế thật ở sau. Bài học: đọc kỹ yêu cầu thiết kế, không chỉ yêu cầu output.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — stack chứa nút thay vì số.** Quy trình y hệt tính postfix, chỉ đổi "tính `left op right`" thành "tạo nút `op` có con `left`, `right`". Invariant: sau mỗi token, stack chứa các **cây con hoàn chỉnh** ứng với các đoạn liên tiếp của postfix đã đọc.

**Insight 2 — đệ quy từ phải sang trái.** Token cuối là gốc. Nếu gốc là toán tử: đọc tiếp (lùi) để dựng **cây con phải** trước, rồi **cây con trái**.

**Insight 3 — tách "dữ liệu" khỏi "hành vi" (follow-up).** Thay vì một `evaluate` dài với chuỗi `if op == '+' ... elif ...`, cho mỗi loại toán tử một lớp (hoặc một hàm trong **bảng đăng ký**). `evaluate` của nút toán tử chỉ làm: `return self.apply(left.evaluate(), right.evaluate())`. Thêm toán tử ⇒ thêm một mục vào bảng / một lớp mới.

Quy tắc "nếu… thì…":
- **Nếu** gặp số **thì** tạo lá, đẩy vào stack.
- **Nếu** gặp toán tử **thì** pop **phải trước**, **trái sau**, tạo nút, đẩy lại.
- **Nếu** cần thêm phép toán mà không sửa code cũ **thì** dùng đa hình / bảng ánh xạ token → lớp.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Một lớp `MyNode` + chuỗi `if` trong `evaluate`, dựng bằng stack

Đơn giản nhất cho phỏng vấn: một lớp duy nhất có `val`, `left`, `right`.

In [7]:
class MyNode(Node):
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def evaluate(self) -> int:
        if self.left is None:                  # lá
            return int(self.val)
        a, b = self.left.evaluate(), self.right.evaluate()
        if self.val == "+": return a + b
        if self.val == "-": return a - b
        if self.val == "*": return a * b
        if self.val == "/": return div_trunc(a, b)
        raise ValueError(self.val)

class TreeBuilderStack:
    def buildTree(self, postfix: List[str]) -> Node:
        st = []
        for tok in postfix:
            if tok in ("+", "-", "*", "/"):
                right = st.pop()
                left = st.pop()
                st.append(MyNode(tok, left, right))
            else:
                st.append(MyNode(tok))
        return st[0]

for s, want in examples:
    print(run_leetcode(TreeBuilderStack, s), "kỳ vọng", want)

2 kỳ vọng 2
-16 kỳ vọng -16


Hai hàm phụ để **nhìn thấy** cây: biểu thức trung tố có ngoặc và postorder (phải trùng đầu vào), cùng một bản vẽ ASCII nghiêng.

In [8]:
def to_infix(node):
    if node.left is None:
        return str(node.val)
    return f"({to_infix(node.left)}{node.val}{to_infix(node.right)})"

def to_postfix(node):
    if node.left is None:
        return [str(node.val)]
    return to_postfix(node.left) + to_postfix(node.right) + [node.val]

def draw(node, prefix="", is_left=True, out=None):
    out = [] if out is None else out
    if node.right is not None:
        draw(node.right, prefix + ("│   " if is_left else "    "), False, out)
    out.append(prefix + ("└── " if is_left else "┌── ") + str(node.val))
    if node.left is not None:
        draw(node.left, prefix + ("    " if is_left else "│   "), True, out)
    return out

root = TreeBuilderStack().buildTree(examples[1][0])
print(to_infix(root)); print(to_postfix(root) == examples[1][0])
print("\n".join(draw(root)))

(4*(5-(2+7)))
True
│           ┌── 7
│       ┌── +
│       │   └── 2
│   ┌── -
│   │   └── 5
└── *
    └── 4


Code nộp LeetCode của Cách 2:

```python
class Node(ABC):
    @abstractmethod
    def evaluate(self) -> int:
        pass

class MyNode(Node):
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

    def evaluate(self) -> int:
        if self.left is None:
            return int(self.val)
        a, b = self.left.evaluate(), self.right.evaluate()
        if self.val == '+': return a + b
        if self.val == '-': return a - b
        if self.val == '*': return a * b
        q = abs(a) // abs(b)                    # '/': làm tròn về 0
        return q if (a >= 0) == (b > 0) else -q

class TreeBuilder(object):
    def buildTree(self, postfix: List[str]) -> 'Node':
        st = []
        for tok in postfix:
            if tok in '+-*/' and len(tok) == 1:
                right, left = st.pop(), st.pop()
                st.append(MyNode(tok, left, right))
            else:
                st.append(MyNode(tok))
        return st[0]
```

**Độ phức tạp:** build `O(n)`, evaluate `O(n)`; Space `O(n)` cho cây + stack (+ đệ quy `O(h)` khi evaluate).

### 7.2 Cách 3 — Dựng đệ quy từ cuối mảng (phải sang trái)

Biến `i` chạy lùi từ token cuối:
1. Lấy `tok = postfix[i]`, `i -= 1`.
2. Nếu `tok` là số ⇒ trả lá.
3. Nếu là toán tử ⇒ dựng **cây phải** (gọi đệ quy), rồi **cây trái**, ghép thành nút.

Thứ tự "phải trước, trái sau" là vì đọc ngược postfix (trái-phải-gốc) ta gặp gốc → phải → trái.

In [9]:
class TreeBuilderRecursive:
    def buildTree(self, postfix: List[str]) -> Node:
        self.i = len(postfix) - 1
        def build():
            tok = postfix[self.i]; self.i -= 1
            if tok in OPS:
                right = build()
                left = build()
                return MyNode(tok, left, right)
            return MyNode(tok)
        root = build()
        assert self.i == -1, "postfix không hợp lệ"
        return root

for s, want in examples:
    r = TreeBuilderRecursive().buildTree(s)
    print(to_infix(r), "=", r.evaluate(), "kỳ vọng", want)

(((3+4)*2)/7) = 2 kỳ vọng 2
(4*(5-(2+7))) = -16 kỳ vọng -16


Code nộp LeetCode của Cách 3 (dùng lại `MyNode` ở Cách 2):

```python
class TreeBuilder(object):
    def buildTree(self, postfix: List[str]) -> 'Node':
        it = iter(reversed(postfix))
        def build():
            tok = next(it)
            if tok in ('+', '-', '*', '/'):
                right = build()
                left = build()
                return MyNode(tok, left, right)
            return MyNode(tok)
        return build()
```

**Độ phức tạp:** Time `O(n)`, Space `O(h)` đệ quy (`h ≤ n/2` vì độ dài `< 100` nên an toàn).

### 7.3 Cách 4 — Thiết kế mô-đun: lớp số + lớp toán tử + **bảng đăng ký** (follow-up)

- `NumNode(value)`: lá, `evaluate` trả `value`.
- `BinaryNode`: lớp trừu tượng cho nút 2 con, `evaluate` = `self.apply(left.evaluate(), right.evaluate())`. Lớp này **không bao giờ cần sửa**.
- Mỗi toán tử là một lớp con chỉ cài `apply`: `Add`, `Sub`, `Mul`, `Div`.
- `REGISTRY = {"+": Add, ...}` — `TreeBuilder` tra bảng để tạo nút. Thêm toán tử mới = viết lớp + `register("%")`.

In [10]:
class NumNode(Node):
    def __init__(self, value: int):
        self.value = value
        self.left = self.right = None
        self.val = str(value)
    def evaluate(self) -> int:
        return self.value

class BinaryNode(Node):
    symbol = "?"
    def __init__(self, left: Node, right: Node):
        self.left, self.right = left, right
        self.val = self.symbol
    def evaluate(self) -> int:
        return self.apply(self.left.evaluate(), self.right.evaluate())
    @abstractmethod
    def apply(self, a: int, b: int) -> int: ...

REGISTRY = {}
def register(symbol):
    def deco(cls):
        cls.symbol = symbol
        REGISTRY[symbol] = cls
        return cls
    return deco

@register("+")
class Add(BinaryNode):
    def apply(self, a, b): return a + b

@register("-")
class Sub(BinaryNode):
    def apply(self, a, b): return a - b

@register("*")
class Mul(BinaryNode):
    def apply(self, a, b): return a * b

@register("/")
class Div(BinaryNode):
    def apply(self, a, b): return div_trunc(a, b)

class TreeBuilderOOP:
    LEAF = NumNode                     # gắn lớp lá ngay lúc định nghĩa
    def buildTree(self, postfix: List[str]) -> Node:
        st = []
        for tok in postfix:
            cls = REGISTRY.get(tok)
            if cls is None:
                st.append(self.LEAF(int(tok)))
            else:
                right = st.pop(); left = st.pop()
                st.append(cls(left, right))
        return st[0]

for s, want in examples:
    r = TreeBuilderOOP().buildTree(s)
    print(to_infix(r), "=", r.evaluate(), "kỳ vọng", want, "| kiểu gốc:", type(r).__name__)

(((3+4)*2)/7) = 2 kỳ vọng 2 | kiểu gốc: Div
(4*(5-(2+7))) = -16 kỳ vọng -16 | kiểu gốc: Mul


**Follow-up trong thực tế:** thêm phép lấy dư `%` và lũy thừa `^` — **không đụng** vào `BinaryNode.evaluate`, `NumNode` hay `TreeBuilderOOP`:

In [11]:
@register("%")
class Mod(BinaryNode):
    def apply(self, a, b): return a % b

@register("^")
class Pow(BinaryNode):
    def apply(self, a, b): return a ** b

expr = ["2", "3", "^", "5", "%", "4", "*"]          # ((2^3) % 5) * 4 = 3 * 4 = 12
r = TreeBuilderOOP().buildTree(expr)
print(to_infix(r), "=", r.evaluate())
assert r.evaluate() == 12
# dọn để các phần sau chỉ dùng 4 toán tử của đề
del REGISTRY["%"], REGISTRY["^"]
print("toán tử đang đăng ký:", sorted(REGISTRY))

(((2^3)%5)*4) = 12
toán tử đang đăng ký: ['*', '+', '-', '/']


Code nộp LeetCode của Cách 4:

```python
class Node(ABC):
    @abstractmethod
    def evaluate(self) -> int:
        pass

class NumNode(Node):
    def __init__(self, value):
        self.value = value
    def evaluate(self) -> int:
        return self.value

class BinaryNode(Node):
    def __init__(self, left, right):
        self.left, self.right = left, right
    def evaluate(self) -> int:
        return self.apply(self.left.evaluate(), self.right.evaluate())
    @abstractmethod
    def apply(self, a, b): ...

class Add(BinaryNode):
    def apply(self, a, b): return a + b
class Sub(BinaryNode):
    def apply(self, a, b): return a - b
class Mul(BinaryNode):
    def apply(self, a, b): return a * b
class Div(BinaryNode):
    def apply(self, a, b):
        q = abs(a) // abs(b)
        return q if (a >= 0) == (b > 0) else -q

class TreeBuilder(object):
    OPS = {'+': Add, '-': Sub, '*': Mul, '/': Div}
    def buildTree(self, postfix: List[str]) -> 'Node':
        st = []
        for tok in postfix:
            if tok in self.OPS:
                right, left = st.pop(), st.pop()
                st.append(self.OPS[tok](left, right))
            else:
                st.append(NumNode(int(tok)))
        return st[0]
```

**Độ phức tạp:** như Cách 2. Thêm toán tử: `O(1)` dòng code mới, **0** dòng sửa.

### 7.4 Cách 5 — `evaluate` không đệ quy (duyệt hậu thứ tự bằng stack)

Với độ dài `< 100` thì đệ quy an toàn, nhưng cây biểu thức trong thực tế có thể sâu hàng chục nghìn tầng (ví dụ `1 1 + 1 + 1 + …`). Ta tính giá trị bằng hậu thứ tự lặp: stack chứa `(nút, đã_xử_lý_con?)`.

In [12]:
def evaluate_iterative(root):
    vals = []                         # stack giá trị
    st = [(root, False)]
    while st:
        node, done = st.pop()
        if node.left is None:
            vals.append(int(node.val)); continue
        if done:
            b = vals.pop(); a = vals.pop()
            vals.append(OPS[node.val](a, b))
        else:
            st.append((node, True))
            st.append((node.right, False))
            st.append((node.left, False))
    return vals[0]

for s, want in examples:
    print(evaluate_iterative(TreeBuilderStack().buildTree(s)), "kỳ vọng", want)

deep = ["1"] + ["1", "+"] * 20000                 # cây lệch sâu 20000 tầng
root_deep = TreeBuilderStack().buildTree(deep)
print("cây sâu 20000:", evaluate_iterative(root_deep))

2 kỳ vọng 2
-16 kỳ vọng -16
cây sâu 20000: 20001


**Độ phức tạp:** Time `O(n)`, Space `O(n)` stack tường minh — không phụ thuộc giới hạn đệ quy của Python.

## 8. TRACE từng bước

Trace **Cách 2** (dựng bằng stack) trên ví dụ 1 `["3","4","+","2","*","7","/"]`:

| token | loại | pop phải | pop trái | nút mới | stack sau |
|---|---|---|---|---|---|
| 3 | số | | | `3` | `[3]` |
| 4 | số | | | `4` | `[3, 4]` |
| + | toán tử | `4` | `3` | `(3+4)` | `[(3+4)]` |
| 2 | số | | | `2` | `[(3+4), 2]` |
| * | toán tử | `2` | `(3+4)` | `((3+4)*2)` | `[((3+4)*2)]` |
| 7 | số | | | `7` | `[((3+4)*2), 7]` |
| / | toán tử | `7` | `((3+4)*2)` | `(((3+4)*2)/7)` | `[…]` |

In [13]:
def trace_build(postfix):
    st = []
    for tok in postfix:
        if tok in OPS:
            right = st.pop(); left = st.pop()
            node = MyNode(tok, left, right)
            print(f"tok={tok!r:4}: pop phải={to_infix(right):12} pop trái={to_infix(left):14} -> {to_infix(node)}")
            st.append(node)
        else:
            st.append(MyNode(tok))
            print(f"tok={tok!r:4}: lá {tok}")
        print("        stack =", [to_infix(x) for x in st])
    return st[0]

root = trace_build(examples[0][0])

tok='3' : lá 3
        stack = ['3']
tok='4' : lá 4
        stack = ['3', '4']
tok='+' : pop phải=4            pop trái=3              -> (3+4)
        stack = ['(3+4)']
tok='2' : lá 2
        stack = ['(3+4)', '2']
tok='*' : pop phải=2            pop trái=(3+4)          -> ((3+4)*2)
        stack = ['((3+4)*2)']
tok='7' : lá 7
        stack = ['((3+4)*2)', '7']
tok='/' : pop phải=7            pop trái=((3+4)*2)      -> (((3+4)*2)/7)
        stack = ['(((3+4)*2)/7)']


In [14]:
def trace_eval(node, depth=0):
    pad = "  " * depth
    if node.left is None:
        print(f"{pad}lá {node.val}")
        return int(node.val)
    print(f"{pad}nút {node.val!r}: tính trái, rồi phải")
    a = trace_eval(node.left, depth + 1)
    b = trace_eval(node.right, depth + 1)
    v = OPS[node.val](a, b)
    print(f"{pad}=> {a} {node.val} {b} = {v}")
    return v

assert trace_eval(root) == 2

nút '/': tính trái, rồi phải
  nút '*': tính trái, rồi phải
    nút '+': tính trái, rồi phải
      lá 3
      lá 4
    => 3 + 4 = 7
    lá 2
  => 7 * 2 = 14
  lá 7
=> 14 / 7 = 2


## 9. Code nộp LeetCode

Bản nộp dùng thiết kế mô-đun (Cách 4) — vừa đúng, vừa trả lời follow-up. Bộ chạy bên dưới mô phỏng chính xác cách LeetCode gọi.

In [15]:
class Node(ABC):
    @abstractmethod
    # define your fields here
    def evaluate(self) -> int:
        pass

class NumNode(Node):
    def __init__(self, value: int):
        self.value = value
    def evaluate(self) -> int:
        return self.value

class BinaryNode(Node):
    def __init__(self, left: Node, right: Node):
        self.left, self.right = left, right
    def evaluate(self) -> int:
        return self.apply(self.left.evaluate(), self.right.evaluate())
    @abstractmethod
    def apply(self, a: int, b: int) -> int: ...

class Add(BinaryNode):
    def apply(self, a, b): return a + b

class Sub(BinaryNode):
    def apply(self, a, b): return a - b

class Mul(BinaryNode):
    def apply(self, a, b): return a * b

class Div(BinaryNode):
    def apply(self, a, b):
        q = abs(a) // abs(b)
        return q if (a >= 0) == (b > 0) else -q

class TreeBuilder(object):
    OPS = {"+": Add, "-": Sub, "*": Mul, "/": Div}

    def buildTree(self, postfix: List[str]) -> 'Node':
        st = []
        for tok in postfix:
            if tok in self.OPS:
                right, left = st.pop(), st.pop()
                st.append(self.OPS[tok](left, right))
            else:
                st.append(NumNode(int(tok)))
        return st[0]

# Driver giống LeetCode
for postfix, want in examples:
    obj = TreeBuilder()
    expTree = obj.buildTree(postfix)
    ans = expTree.evaluate()
    print(postfix, "->", ans, "OK" if ans == want else "SAI")

['3', '4', '+', '2', '*', '7', '/'] -> 2 OK
['4', '5', '2', '7', '+', '-', '*'] -> -16 OK


## 10. Vì sao thuật toán đúng

**Dựng bằng stack.** Invariant: sau khi đọc tiền tố `postfix[0..i]`, stack (đáy → đỉnh) là các cây `T₁, …, T_m` sao cho postorder của `T₁` nối … nối postorder của `T_m` **bằng đúng** tiền tố đã đọc, và mỗi `Tⱼ` là cây biểu thức hợp lệ.
- Token số: thêm lá — invariant giữ.
- Token toán tử `op`: trong biểu thức hợp lệ, hai toán hạng của `op` là hai biểu thức con **liền kề ngay trước** `op` ⇒ chính là `T_{m−1}` (trái) và `T_m` (phải). Ghép thành `(T_{m−1} op T_m)` có postorder = postorder(`T_{m−1}`) + postorder(`T_m`) + `[op]` — invariant giữ.
- Kết thúc: postfix hợp lệ ⇒ còn đúng một cây, postorder của nó là toàn bộ đầu vào ⇒ đó là cây của biểu thức.

**Evaluate.** Quy nạp theo chiều cao: lá trả đúng số; nút `op` trả `op(giá trị trái, giá trị phải)` = đúng định nghĩa giá trị biểu thức.

**Đệ quy phải-trước-trái (Cách 3)** đúng vì postorder đọc ngược là "gốc, (postorder phải đọc ngược), (postorder trái đọc ngược)".

**Thiết kế mô-đun.** `BinaryNode.evaluate` chỉ phụ thuộc vào hợp đồng `apply(a, b)`; mọi lớp con thỏa hợp đồng đều được tính đúng ⇒ mở rộng mà không sửa.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(postfix)` (< 100), `h` = chiều cao cây (`≤ (n−1)/2`).

| Cách | buildTree | evaluate | Space |
|---|---|---|---|
| 1. Baseline (tính sẵn) | `O(n)` | `O(1)` | `O(n)` stack |
| 2. MyNode + stack | `O(n)` | `O(n)` | `O(n)` cây, `O(h)` đệ quy |
| 3. Đệ quy từ phải | `O(n)` | `O(n)` | `O(h)` đệ quy khi build |
| 4. OOP + registry | `O(n)` | `O(n)` | như Cách 2 |
| 5. evaluate lặp | `O(n)` | `O(n)` | `O(n)` stack tường minh |

Số nút cây = `n` (mỗi token một nút). Với số nguyên Python, các phép tính trên giá trị `≤ 10⁹` là `O(1)`.

## 12. Bộ test

- Ví dụ đề; một số duy nhất `["7"]`; phép trừ/chia **không giao hoán** (`["8","2","-"]`, `["8","2","/"]`) để bắt lỗi thứ tự pop; chia ra số âm.
- Sinh cây ngẫu nhiên (seed cố định), đảm bảo **mọi giá trị trung gian `≤ 10⁹`**, không chia cho 0, số `≤ 10⁵`, độ dài `< 100`; xuất postfix; so tất cả TreeBuilder với `eval_postfix`; kiểm tra postorder của cây trùng đầu vào và mọi nút có 0 hoặc 2 con.

In [16]:
BUILDERS = [TreeBuilderBaseline, TreeBuilderStack, TreeBuilderRecursive, TreeBuilderOOP, TreeBuilder]
fixed = examples + [
    (["7"], 7),
    (["8", "2", "-"], 6),
    (["8", "2", "/"], 4),
    (["2", "8", "-"], -6),
    (["0", "5", "/"], 0),
    (["2", "9", "-", "2", "/"], -3),        # (2-9)/2 = -7/2 -> làm tròn về 0 = -3
    (["100000", "100000", "*", "10", "/"], 1000000000),
]
for s, want in fixed:
    for B in BUILDERS:
        assert run_leetcode(B, s) == want, (B.__name__, s)
    assert eval_postfix(s) == want
    assert evaluate_iterative(TreeBuilderStack().buildTree(s)) == want
print("ví dụ cố định: OK")

ví dụ cố định: OK


In [17]:
def random_postfix(rnd, max_len=99, max_num=10**5, limit=10**9):
    """Sinh postfix hợp lệ ngẫu nhiên thỏa ràng buộc đề."""
    while True:
        leaves = rnd.randint(1, (max_len + 1) // 2)
        def gen(k):                      # trả (tokens, value) cho cây có k lá
            if k == 1:
                v = rnd.randint(0, rnd.choice([9, 100, max_num]))
                return [str(v)], v
            kl = rnd.randint(1, k - 1)
            lt, lv = gen(kl); rt, rv = gen(k - kl)
            if lv is None or rv is None:
                return [], None
            op = rnd.choice("+-*/")
            if op == "/" and rv == 0:
                op = rnd.choice("+-*")
            v = OPS[op](lv, rv)
            if abs(v) > limit:
                return [], None
            return lt + rt + [op], v
        toks, val = gen(leaves)
        if val is not None and len(toks) < 100:
            return toks, val

def check_shape(node):
    """Mọi nút có 0 hoặc 2 con."""
    l, r = getattr(node, "left", None), getattr(node, "right", None)
    if (l is None) != (r is None):
        return False
    return l is None or (check_shape(l) and check_shape(r))

rnd = random.Random(1628)
for _ in range(500):
    toks, val = random_postfix(rnd)
    assert eval_postfix(toks) == val
    for B in BUILDERS:
        assert run_leetcode(B, toks) == val, (B.__name__, toks)
    for B in (TreeBuilderStack, TreeBuilderRecursive, TreeBuilderOOP):
        root = B().buildTree(toks)
        assert to_postfix(root) == toks and check_shape(root)
        assert evaluate_iterative(root) == val
print("random tests: OK")

random tests: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Pop sai thứ tự toán hạng (trái trước, phải sau)

In [18]:
class TreeBuilderWrongOrder:
    def buildTree(self, postfix):
        st = []
        for tok in postfix:
            if tok in OPS:
                left = st.pop(); right = st.pop()          # SAI: phần tử pop đầu tiên là toán hạng PHẢI
                st.append(MyNode(tok, left, right))
            else:
                st.append(MyNode(tok))
        return st[0]

for s in (["8", "2", "-"], examples[0][0]):
    print(s, "sai:", run_leetcode(TreeBuilderWrongOrder, s), "| đúng:", eval_postfix(s))

['8', '2', '-'] sai: -6 | đúng: 6
['3', '4', '+', '2', '*', '7', '/'] sai: 0 | đúng: 2


**Vì sao sai:** đỉnh stack là toán hạng được đẩy **sau cùng**, tức toán hạng **phải**. Với `+`, `*` (giao hoán) lỗi bị che; với `-`, `/` lộ ngay: `8 2 -` thành `2 − 8`. **Sửa:** `right = st.pop(); left = st.pop()`. Luôn test bằng phép không giao hoán.

### Lỗi 2 — Dùng `//` của Python cho phép chia khi có số âm

In [19]:
class MyNodeFloorDiv(MyNode):
    def evaluate(self):
        if self.left is None:
            return int(self.val)
        a, b = self.left.evaluate(), self.right.evaluate()
        if self.val == "/":
            return a // b                                  # làm tròn XUỐNG
        return OPS[self.val](a, b)

def build_with(node_cls, postfix):
    st = []
    for tok in postfix:
        if tok in OPS:
            r = st.pop(); l = st.pop(); st.append(node_cls(tok, l, r))
        else:
            st.append(node_cls(tok))
    return st[0]

s = ["2", "9", "-", "2", "/"]                               # (2-9)/2 = -7/2
print("// cho:", build_with(MyNodeFloorDiv, s).evaluate(), "| làm tròn về 0:", eval_postfix(s))

// cho: -4 | làm tròn về 0: -3


**Vì sao sai (có thể):** `-7 // 2 = -4` (làm tròn xuống), còn chia nguyên kiểu C++/Java cho `-3`. Nếu bộ test của LeetCode có phép chia không chẵn với số âm, hai cách cho kết quả khác nhau. **Sửa an toàn:** `int(a / b)` (giá trị `≤ 10⁹` nên float đủ chính xác) hoặc `div_trunc` dùng số nguyên thuần. *Đối chiếu lại đề gốc để biết chính xác quy ước.*

### Lỗi 3 — Nhận diện số bằng `isdigit()` (hỏng với số âm) hoặc so sánh ký tự đầu

In [20]:
def is_operator_wrong(tok):
    return not tok.isdigit()                  # SAI: "-3".isdigit() == False -> bị coi là toán tử

def is_operator_wrong2(tok):
    return tok[0] in "+-*/"                   # SAI: "-3"[0] == "-"

for tok in ["12", "-3", "-", "+"]:
    print(f"{tok!r:5}: isdigit-cách={is_operator_wrong(tok)}, ký-tự-đầu={is_operator_wrong2(tok)}, đúng={tok in OPS}")

'12' : isdigit-cách=False, ký-tự-đầu=False, đúng=False
'-3' : isdigit-cách=True, ký-tự-đầu=True, đúng=False
'-'  : isdigit-cách=True, ký-tự-đầu=True, đúng=True
'+'  : isdigit-cách=True, ký-tự-đầu=True, đúng=True


**Vì sao sai:** đề hiện tại chỉ có số không âm, nhưng các biến thể (và nhiều bài postfix khác như LC 150) có số âm `"-3"`. Cả hai cách trên coi `"-3"` là toán tử ⇒ pop sai, stack rỗng. **Sửa:** kiểm tra `tok in {"+", "-", "*", "/"}` (so **cả chuỗi**).

### Lỗi 4 — Tạo đối tượng từ lớp trừu tượng hoặc quên cài `evaluate` ở lớp con

In [21]:
class LeafForgot(Node):            # quên cài evaluate
    def __init__(self, v): self.v = v

for make in (lambda: Node(), lambda: LeafForgot(3)):
    try:
        make()
        print("tạo được (?)")
    except TypeError as e:
        print("TypeError:", str(e).split(" with")[0])

TypeError: Can't instantiate abstract class Node
TypeError: Can't instantiate abstract class LeafForgot


**Vì sao sai:** `Node` có `@abstractmethod` nên không khởi tạo được; lớp con chưa cài `evaluate` cũng vẫn là trừu tượng. **Sửa:** mỗi lớp cụ thể phải định nghĩa `evaluate` (hoặc kế thừa từ lớp cha đã cài, như `BinaryNode`).

### Lỗi 5 — Dùng `eval()` trên chuỗi trung tố

In [22]:
def wrong_eval(postfix):
    root = TreeBuilderStack().buildTree(postfix)
    return eval(to_infix(root))                 # SAI: '/' của Python là chia số thực

for s in (examples[0][0], ["7", "2", "/"]):
    print(s, "eval():", wrong_eval(s), "| đúng:", eval_postfix(s))

['3', '4', '+', '2', '*', '7', '/'] eval(): 2.0 | đúng: 2
['7', '2', '/'] eval(): 3.5 | đúng: 3


**Vì sao sai:** `/` trong Python cho số thực (`7/2 = 3.5`, `14/7 = 2.0` — sai kiểu `int`); ngoài ra `eval` trên chuỗi tùy ý là **lỗ hổng bảo mật** và không phải "thiết kế cây". **Sửa:** tự cài `evaluate` theo cây.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **postfix / RPN** ⇒ stack; "dựng cây" ⇒ stack chứa **nút** thay vì **số**.
- Thứ tự pop: **phải trước, trái sau** — viết một test với `-` hoặc `/` ngay từ đầu.
- Bài **Design** trên LeetCode: đọc kỹ "driver" gọi gì (`TreeBuilder().buildTree(...).evaluate()`), viết đúng tên lớp/method.
- Follow-up "thêm toán tử không sửa code" ⇒ **đa hình** + **bảng đăng ký** (Strategy / Factory pattern).
- Chia nguyên trong Python: luôn hỏi "làm tròn xuống hay về 0?".

## 15. Biến thể của bài

1. **LC 150 Evaluate Reverse Polish Notation** — chỉ tính giá trị (có số âm, chia làm tròn về 0).
2. **LC 1597 Build Binary Expression Tree From Infix Expression** (Hard) — dựng cây từ trung tố (cần độ ưu tiên + ngoặc).
3. **LC 1612 Check If Two Expression Trees are Equivalent** — so sánh hai cây chỉ có `+`.
4. **LC 224 / 227 Basic Calculator I/II** — trung tố với ngoặc, độ ưu tiên.
5. **Toán tử một ngôi** (ví dụ `neg`) — thêm lớp `UnaryNode` với một con; builder pop 1 phần tử. Thiết kế mô-đun xử lý tự nhiên.
6. **In biểu thức với số ngoặc tối thiểu** — dựa vào độ ưu tiên của nút cha/con.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Baseline tính sẵn | stack số, nút giữ kết quả | `O(n)` | `O(n)` | oracle; **không** đạt yêu cầu thiết kế |
| 2. MyNode + stack | 1 lớp, `if` theo toán tử | `O(n)` | `O(n)` | nhanh gọn khi phỏng vấn |
| 3. Đệ quy từ phải | gốc → phải → trái | `O(n)` | `O(h)` | khi thích tư duy đệ quy |
| 4. OOP + registry | đa hình, mỗi toán tử 1 lớp | `O(n)` | `O(n)` | trả lời follow-up, code thực tế |
| 5. evaluate lặp | hậu thứ tự bằng stack | `O(n)` | `O(n)` | cây rất sâu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** giá trị của `["5","1","2","+","4","*","+","3","-"]`, `["6","3","/","2","/"]`, `["2","3","4","*","-"]`.

**Bài 2 (tính tay):** vẽ cây của `["1","2","+","3","4","-","*"]`, viết dạng trung tố có ngoặc, và tính giá trị.

**Bài 3 (điền TODO):** viết `count_nodes(root)` và `height(root)` cho cây dựng từ `TreeBuilderStack` (lá có chiều cao 0).

**Bài 4 (tự viết lại):** viết `infix_to_postfix(tokens)` cho biểu thức trung tố có `+ - * /` và ngoặc (thuật toán **shunting-yard**), rồi dùng `TreeBuilder` để tính.

**Bài 5 (mini challenge):** mở rộng thiết kế mô-đun với toán tử **một ngôi** `"neg"` (đổi dấu) — thêm lớp mới và một builder mở rộng **không sửa** `NumNode`, `BinaryNode`; kiểm tra `["3","neg","4","+"]` = `1`.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_nodes(root):
    # TODO: lá -> 1; nút trong -> 1 + trái + phải
    pass

def height(root):
    # TODO: lá -> 0
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def infix_to_postfix(tokens):
    # TODO: shunting-yard
    pass

class NegNode(Node):
    # TODO
    def evaluate(self) -> int:
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: dùng stack số. Mức 2: nhớ pop phải trước. Mức 3: `5 + (1+2)*4 − 3 = 14`; `(6/3)/2 = 1`; `2 − 3·4 = −10`.

**Bài 2.** Mức 1: token cuối `*` là gốc. Mức 2: cây trái `(1+2)`, cây phải `(3−4)`. Mức 3: `((1+2)*(3-4)) = −3`.

**Bài 3.** Mức 1: đệ quy theo định nghĩa. Mức 2: lá nhận ra bằng `left is None`. Mức 3: `count_nodes` luôn bằng `len(postfix)`; cây ví dụ 1 có chiều cao 3.

**Bài 4.** Mức 1: hai stack: output và toán tử. Mức 2: gặp toán tử, pop các toán tử trên đỉnh có độ ưu tiên `≥` (vì `+ - * /` kết hợp trái). Mức 3: `(` đẩy vào; `)` pop tới khi gặp `(`.

**Bài 5.** Mức 1: `NegNode(child)` có `evaluate = -child.evaluate()`. Mức 2: builder: bảng `UNARY = {"neg": NegNode}` pop 1, bảng `BINARY` pop 2. Mức 3: hàm `to_postfix`/`check_shape` cần chấp nhận nút 1 con — đó là lý do đề gốc chỉ có toán tử hai ngôi.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = [run_leetcode(TreeBuilder, s) for s in (["5", "1", "2", "+", "4", "*", "+", "3", "-"], ["6", "3", "/", "2", "/"], ["2", "3", "4", "*", "-"])]
print("Bài 1:", b1)
assert b1 == [14, 1, -10]
r2 = TreeBuilderStack().buildTree(["1", "2", "+", "3", "4", "-", "*"])
print("Bài 2:", to_infix(r2), "=", r2.evaluate())
print("\n".join(draw(r2)))
assert to_infix(r2) == "((1+2)*(3-4))" and r2.evaluate() == -3

Bài 1: [14, 1, -10]
Bài 2: ((1+2)*(3-4)) = -3
│       ┌── 4
│   ┌── -
│   │   └── 3
└── *
    │   ┌── 2
    └── +
        └── 1


In [26]:
# Bài 3
def count_nodes_sol(root):
    return 1 if root.left is None else 1 + count_nodes_sol(root.left) + count_nodes_sol(root.right)

def height_sol(root):
    return 0 if root.left is None else 1 + max(height_sol(root.left), height_sol(root.right))

assert height_sol(TreeBuilderStack().buildTree(examples[0][0])) == 3
rnd = random.Random(3)
for _ in range(200):
    toks, _ = random_postfix(rnd)
    r = TreeBuilderStack().buildTree(toks)
    assert count_nodes_sol(r) == len(toks)
    assert height_sol(r) <= (len(toks) - 1) // 2
print("Bài 3: OK")

Bài 3: OK


In [27]:
# Bài 4 — shunting-yard
PREC = {"+": 1, "-": 1, "*": 2, "/": 2}

def infix_to_postfix_sol(tokens):
    out, ops = [], []
    for t in tokens:
        if t in PREC:
            while ops and ops[-1] in PREC and PREC[ops[-1]] >= PREC[t]:
                out.append(ops.pop())
            ops.append(t)
        elif t == "(":
            ops.append(t)
        elif t == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()
        else:
            out.append(t)
    while ops:
        out.append(ops.pop())
    return out

def tokenize(expr):
    toks, num = [], ""
    for ch in expr.replace(" ", ""):
        if ch.isdigit():
            num += ch
        else:
            if num: toks.append(num); num = ""
            toks.append(ch)
    if num: toks.append(num)
    return toks

for expr, want in [("4*(5-(7+2))", -16), ("((3+4)*2)/7", 2), ("2-3-4", -5), ("8/2/2", 2), ("1+2*3-4", 3)]:
    pf = infix_to_postfix_sol(tokenize(expr))
    got = run_leetcode(TreeBuilder, pf)
    print(f"{expr:14} -> {pf} -> {got}")
    assert got == want
rnd = random.Random(4)
for _ in range(300):
    toks, val = random_postfix(rnd)
    inf = tokenize(to_infix(TreeBuilderStack().buildTree(toks)))
    assert run_leetcode(TreeBuilder, infix_to_postfix_sol(inf)) == val
print("Bài 4: OK")

4*(5-(7+2))    -> ['4', '5', '7', '2', '+', '-', '*'] -> -16
((3+4)*2)/7    -> ['3', '4', '+', '2', '*', '7', '/'] -> 2
2-3-4          -> ['2', '3', '-', '4', '-'] -> -5
8/2/2          -> ['8', '2', '/', '2', '/'] -> 2
1+2*3-4        -> ['1', '2', '3', '*', '+', '4', '-'] -> 3
Bài 4: OK


In [28]:
# Bài 5 — toán tử một ngôi, không sửa NumNode / BinaryNode
class NegNode(Node):
    def __init__(self, child: Node):
        self.child = child
    def evaluate(self) -> int:
        return -self.child.evaluate()

class TreeBuilderExt(TreeBuilder):
    UNARY = {"neg": NegNode}
    def buildTree(self, postfix):
        st = []
        for tok in postfix:
            if tok in self.UNARY:
                st.append(self.UNARY[tok](st.pop()))
            elif tok in self.OPS:
                right, left = st.pop(), st.pop()
                st.append(self.OPS[tok](left, right))
            else:
                st.append(NumNode(int(tok)))
        return st[0]

assert TreeBuilderExt().buildTree(["3", "neg", "4", "+"]).evaluate() == 1
assert TreeBuilderExt().buildTree(["2", "5", "-", "neg", "3", "*"]).evaluate() == 9
for s, want in examples:                                # vẫn tương thích ngược
    assert TreeBuilderExt().buildTree(s).evaluate() == want
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template dựng cây từ postfix:**

```python
st = []
for tok in postfix:
    if tok in OPS:                 # so cả chuỗi
        right = st.pop()           # pop PHẢI trước
        left = st.pop()
        st.append(OPS[tok](left, right))   # tạo nút toán tử
    else:
        st.append(NumNode(int(tok)))
return st[0]
```

**Template thiết kế mở rộng:** `Node (ABC)` → `NumNode`, `BinaryNode(evaluate = apply(L, R))` → `Add/Sub/Mul/Div(apply)`; builder tra bảng `{"+": Add, ...}`.

**Checklist:**
- [ ] Đúng tên `Node`, `TreeBuilder`, `buildTree(self, postfix)`, `evaluate(self)`?
- [ ] Pop **phải trước, trái sau**? Đã test `8 2 -` → 6, `8 2 /` → 4?
- [ ] Phép chia: làm tròn về 0 hay xuống? (đối chiếu đề gốc)
- [ ] Nhận diện toán tử bằng **so sánh cả chuỗi**, không `isdigit()`?
- [ ] Mọi lớp cụ thể đã cài `evaluate`?
- [ ] Test một số duy nhất `["7"]`?